# Interactive ROI profile extraction

This notebook lets you load a DNG image, draw a bounding box interactively, and compute the average/median intensity profile across the selected region.

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rawpy
from matplotlib.widgets import RectangleSelector
from scipy.signal import savgol_filter

%matplotlib widget

data_dir = Path('data/led_profile_20260721_135455_45cm_fiducial/y_axis')
image_path = data_dir / '20260721_122317_953_cam_2_raw_sensor_dng_2592x1944.dng'

def load_image(path):
    raw = rawpy.imread(str(path))
    try:
        rgb = raw.postprocess(output_color=rawpy.ColorSpace.sRGB, no_auto_bright=True, output_bps=16, use_camera_wb=True)
    except TypeError:
        rgb = raw.postprocess()
    raw.close()
    image = np.asarray(rgb, dtype=np.float32)
    if image.ndim == 3:
        return np.mean(image, axis=2)
    return image

image = load_image(image_path)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(image, cmap='gray', origin='lower')
ax.set_title('Draw a rectangle around the ROI and press Enter')

bbox = None
rect_patch = None

def onselect(eclick, erelease):
    global bbox, rect_patch
    x0 = int(round(eclick.xdata))
    x1 = int(round(erelease.xdata))
    y0 = int(round(eclick.ydata))
    y1 = int(round(erelease.ydata))
    x0, x1 = sorted((x0, x1))
    y0, y1 = sorted((y0, y1))
    bbox = (max(0, x0), min(image.shape[1]-1, x1), max(0, y0), min(image.shape[0]-1, y1))
    if rect_patch is not None:
        rect_patch.remove()
    rect_patch = plt.Rectangle((bbox[0], bbox[2]), bbox[1]-bbox[0], bbox[3]-bbox[2], fill=False, edgecolor='red', linewidth=2)
    ax.add_patch(rect_patch)
    fig.canvas.draw_idle()

def on_key(event):
    global bbox
    if event.key == 'enter' and bbox is not None:
        plt.close(fig)

fig.canvas.mpl_connect('key_press_event', on_key)
selector = RectangleSelector(ax, onselect, interactive=True, useblit=False, button=[1], minspanx=5, minspany=5, spancoords='pixels')
plt.show()

if bbox is None:
    raise RuntimeError('No bounding box selected')

x0, x1, y0, y1 = bbox
roi = image[y0:y1+1, x0:x1+1]
profile = np.mean(roi, axis=0)
smoothed = savgol_filter(profile, window_length=min(11, len(profile) if len(profile)%2==1 else len(profile)-1), polyorder=3)
gradient = np.gradient(smoothed)
x = np.arange(len(profile))

fig2, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(x, profile, label='Mean profile')
axes[0].plot(x, smoothed, label='Smoothed')
axes[0].legend()
axes[0].set_ylabel('Intensity')
axes[1].plot(x, gradient, color='tab:red')
axes[1].set_ylabel('Gradient')
axes[1].set_xlabel('x (pixels)')
plt.tight_layout()
plt.show()